# Saturation flagging

A source saturates when its **brightest pixel** reaches the detector's ADC full scale
(`adc_max = 2**bit_depth - 1`, in ADU). The peak-pixel value combines the source PSF peak,
the per-pixel sky background, the dark current, and the additive bias.

Key API: `sensor.bit_depth` / `bias_level` / `adc_max`, `sim.peak_pixel_fraction()`,
`sim.get_peak_pixel(time, units='adu')`, and `sim.is_saturated(time)`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import wcc_etc

wcc_etc.set_wcc_style()

scene = wcc_etc.get_scene(
    name="G5V",
    mag=15,
    background="zodi",
    bandpass="johnson_r",
    background_prop={"bandpass": "johnson_r", "mag": 22.5},
)
sim = wcc_etc.Simulation.from_sensor_and_scene("sony:r", scene)

## What the detector can hold

`adc_max` is the clip ceiling in ADU. `peak_pixel_fraction()` is the fraction of the total
source energy landing in the single brightest pixel — it depends on wavelength, f-number,
pixel size, and jitter, and it is what turns a source magnitude into a peak-pixel value.

In [ ]:
print(f"bit depth           : {sim.sensor.bit_depth}")
print(f"adc_max             : {sim.sensor.adc_max}")
print(f"bias level          : {sim.sensor.bias_level}")
print(f"peak pixel fraction : {sim.peak_pixel_fraction():.4f}")

## What saturation looks like

`ImageSimulator.simulate` returns a `SimulatedImage` carrying a `saturation_mask`, and
`plot_image_row` shows the noisy image, the noiseless image, and that mask on a shared
colour scale. At 5 s this r = 12 star is comfortably within range; at 60 s the core clips.

In [ ]:
bright = wcc_etc.get_scene(
    name="G5V",
    mag=12,
    background="zodi",
    bandpass="johnson_r",
    background_prop={"bandpass": "johnson_r", "mag": 22.5},
)
imsim = wcc_etc.ImageSimulator.from_sensor_and_scene("sony:r", bright, npix=96)

for t in (5, 60):
    res = imsim.simulate(time=t, seed=0)
    fig, axes = res.plot_image_row(stretch="log", units="mas", figsize=(13, 4))
    fig.suptitle(
        f"G5V r=12, sony:r, {t} s — {res.saturation_mask.sum()} saturated pixel(s)",
        y=1.02,
    )
    fig.tight_layout()
    plt.show()

## Onset vs exposure time

`get_peak_pixel` and `is_saturated` broadcast over an array of times, so the whole ramp to
the clip ceiling comes from one call each.

On `sony:r` this G5V star saturates after **~0.4 s at r = 12**, **~6.4 s at r = 15**
and **~104 s at r = 18** — about 16x more time per 3 magnitudes, which is just Pogson
scaling (2.512³ ≈ 15.9) showing up in the saturation limit.

In [ ]:
texp = np.logspace(-1, 3, 200)  # 0.1 s to 1000 s
adc_max = sim.sensor.adc_max.value

fig, ax = plt.subplots(figsize=(7, 4.2))
for mag in (12, 15, 18):
    sim.update(source__mag=mag)
    peak = sim.get_peak_pixel(texp, units="adu").value
    sat = sim.is_saturated(texp)
    t_sat = texp[sat][0] if sat.any() else None
    note = f"  (saturates ~{t_sat:.2g} s)" if t_sat else "  (no saturation)"
    ax.loglog(texp, peak, label=f"r = {mag}{note}")

ax.axhline(adc_max, color="crimson", ls="--", label=f"adc_max = {adc_max:.0f} ADU")
ax.set_xlabel("Exposure time [s]")
ax.set_ylabel("Peak pixel [ADU]")
ax.set_title("Saturation onset vs source brightness (sony:r)")
ax.legend(fontsize=9)
plt.show()

sim.update(source__mag=15)

## Bit depth sets the ceiling

The qcmos sensor is 12-bit (`adc_max = 4095`), so it clips far sooner than the 16-bit
Sony for the same star — a lower full scale, and a different gain.

At r = 15 the 16-bit Sony clips after **~6.4 s**; the 12-bit qcmos, whose full scale is
**4095 ADU** rather than 65535, clips after **~0.12 s** — roughly **56x sooner** for the
same star.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for label, s in [
    ("sony:r (16-bit)", sim),
    ("qcmos:r (12-bit)", wcc_etc.Simulation.from_sensor_and_scene("qcmos:r", scene)),
]:
    peak = s.get_peak_pixel(texp, units="adu").value
    sat = s.is_saturated(texp)
    t_sat = texp[sat][0] if sat.any() else None
    (line,) = ax.loglog(texp, peak, label=label)
    ax.axhline(s.sensor.adc_max.value, color=line.get_color(), ls="--", lw=1)
    if t_sat:
        ax.axvline(t_sat, color=line.get_color(), ls=":", lw=1)

ax.set_xlabel("Exposure time [s]")
ax.set_ylabel("Peak pixel [ADU]")
ax.set_title("16-bit vs 12-bit full scale (G5V, r = 15)")
ax.legend(fontsize=9)
plt.show()

## Summary

- A source saturates when its peak pixel reaches `sensor.adc_max`; the budget is
  source PSF peak + sky + dark + bias.
- `sim.peak_pixel_fraction()` converts a source magnitude into a peak-pixel value.
- `sim.get_peak_pixel(time, units='adu')` and `sim.is_saturated(time)` both broadcast
  over an array of times.
- `SimulatedImage.saturation_mask` marks the clipped pixels, and `plot_image_row` draws it.
- Bit depth sets the ceiling: the 12-bit qcmos clips well before the 16-bit Sony.
- Saturation is evaluated **per frame** — see `05_n_reads_exptime.ipynb` for `n_reads`.